In [70]:
print("this is maze runner core mechanics")

this is maze runner core mechanics


In [71]:
# Maze Runner Core Mechanics
# maze creation

In [72]:
import random
import numpy as np

class Gridworld:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        # 0 = free, 1 = obstacle
        self.grid = np.zeros((height, width), dtype=int)
        self.start = (0, 0)
        self.goal = (height - 1, width - 1)

    def random_obstacles(self, probability=0.25):
        for i in range(self.height):
            for j in range(self.width):
                if (i, j) == self.start or (i, j) == self.goal:
                    continue
                self.grid[i][j] = 1 if random.random() < probability else 0

    def is_walkable(self, position):
        x, y = position
        return self.grid[x][y] == 0

    def get_neighbors(self, position):
        x, y = position
        directions = [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1)
        ]
        neighbors = []
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.height and 0 <= ny < self.width:
                if self.is_walkable((nx, ny)):
                    neighbors.append((nx, ny))
        return neighbors

    def display(self, path=None):
        path = set(path or [])
        for i in range(self.height):
            row = []
            for j in range(self.width):
                position = (i, j)
                if position == self.start:
                    row.append("S")
                elif position == self.goal:
                    row.append("G")
                elif position in path:
                    row.append("*")
                elif self.grid[i][j] == 1:
                    row.append("#")
                else:
                    row.append(".")
            print(" ".join(row))

In [73]:
from dataclasses import dataclass

@dataclass
class SearchStats:
    nodes_expanded: int = 0
    nodes_discovered: int = 0
    max_frontier_size: int = 0
    path_length: int = 0
    execution_time: float = 0.0
    path_found: bool = False

In [ ]:
from collections import deque
import time

class Search:
    def __init__(self, environment):
        self.environment = environment
        self.stats = SearchStats()
        self.traversal_order = []

    def bfs(self):
        self.stats = SearchStats()
        self.traversal_order = []

        start_time = time.perf_counter()

        queue = deque([self.environment.start])
        visited = {self.environment.start}

        parent = {
            self.environment.start: None
        }

        self.stats.nodes_discovered = 1

        while queue:

            self.stats.max_frontier_size = max(
                self.stats.max_frontier_size,
                len(queue)
            )

            current = queue.popleft()

            # Record actual BFS traversal order
            self.traversal_order.append(current)

            self.stats.nodes_expanded += 1

            if current == self.environment.goal:
                path = self.reconstruct_path(parent)

                self.stats.path_found = True
                self.stats.path_length = len(path)
                self.stats.execution_time = (
                    time.perf_counter() - start_time
                )

                return path

            for neighbor in self.environment.get_neighbors(current):

                if neighbor in visited:
                    continue

                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)

                self.stats.nodes_discovered += 1

        self.stats.execution_time = (
            time.perf_counter() - start_time
        )

        return None

    def reconstruct_path(self, parent):
        path = []

        current = self.environment.goal

        while current is not None:
            path.append(current)
            current = parent[current]

        path.reverse()

        return path

    def print_stats(self):
        stats = self.stats

        print("\n" + "=" * 40)
        print("           SEARCH STATISTICS")
        print("=" * 40)

        print(f"  Path found       : {'YES' if stats.path_found else 'NO'}")
        print(f"  Path length      : {stats.path_length}")
        print(f"  Nodes expanded   : {stats.nodes_expanded}")
        print(f"  Nodes discovered : {stats.nodes_discovered}")
        print(f"  Max frontier     : {stats.max_frontier_size}")
        print(f"  Execution time   : {stats.execution_time * 1000:.3f} ms")

        print("=" * 40)
            
    def print_traversal(self):
        traversal = {
            position: step
            for step, position in enumerate(self.traversal_order, start=1)
        }

        print("\n" + "=" * 50)
        print("              BFS TRAVERSAL")
        print("=" * 50)

        for i in range(self.environment.height):
            row = []

            for j in range(self.environment.width):
                position = (i, j)

                if position == self.environment.start:
                    cell = " S "
                elif position == self.environment.goal:
                    cell = " G "
                elif self.environment.grid[i][j] == 1:
                    cell = " # "
                elif position in traversal:
                    cell = f"{traversal[position]:2} "
                else:
                    cell = " . "

                row.append(cell)

            print(" ".join(row))

        print("=" * 50)

In [76]:
grid = Gridworld(10, 10)
grid.random_obstacles(probability=0.25)
search = Search(grid)
path = search.bfs()
if path:
    grid.display(path)
    print("Path length:", len(path))
else:
    print("No path found")
search.print_stats()
search.print_traversal()

S . . . . # . # . .
* . . . . . . . . #
* # . # . . . # . .
* * * * . # # . # .
. . # * . . . . . .
. . # * * . . . . .
# # . # * * # . . .
. . # . # * . . . #
. # . . . * # . . .
. . . . . * * * * G
Path length: 19

           SEARCH STATISTICS
  Path found       : YES
  Path length      : 19
  Nodes expanded   : 75
  Nodes discovered : 76
  Max frontier     : 7
  Execution time   : 0.128 ms

           BFS TRAVERSAL ORDER
    1 → (0, 0)
    2 → (1, 0)
    3 → (0, 1)
    4 → (2, 0)
    5 → (1, 1)
    6 → (0, 2)
    7 → (3, 0)
    8 → (1, 2)
    9 → (0, 3)
   10 → (4, 0)
   11 → (3, 1)
   12 → (2, 2)
   13 → (1, 3)
   14 → (0, 4)
   15 → (5, 0)
   16 → (4, 1)
   17 → (3, 2)
   18 → (1, 4)
   19 → (5, 1)
   20 → (3, 3)
   21 → (2, 4)
   22 → (1, 5)
   23 → (4, 3)
   24 → (3, 4)
   25 → (2, 5)
   26 → (1, 6)
   27 → (5, 3)
   28 → (4, 4)
   29 → (2, 6)
   30 → (0, 6)
   31 → (1, 7)
   32 → (5, 4)
   33 → (4, 5)
   34 → (1, 8)
   35 → (6, 4)
   36 → (5, 5)
   37 → (4, 6)
   38 → (0, 8)
  